# Notebook 04 — Gold Layer: Agregações e Modelo Analítico
## Projeto: Otimização de Frotas Citi Bike NYC
### Integrante 3 — Medallion Architecture & Storage

---

**Objetivo:** Construir a camada **Gold** do Data Lake — tabelas analíticas otimizadas para consumo por modelos de ML, dashboards e relatórios. Todos os artefatos são escritos em **Delta Lake** com `OPTIMIZE + ZORDER` para máxima performance de leitura.

**Modelo Estrela implementado:**

```
                    ┌─────────────┐
                    │  dim_time   │
                    └──────┬──────┘
                           │
┌──────────────┐   ┌───────▼──────────┐   ┌──────────────────────┐
│ dim_stations │───│   fact_trips     │───│  agg_demand_hourly   │
└──────────────┘   └───────┬──────────┘   └──────────────────────┘
                           │               ┌──────────────────────┐
                           └───────────────│  agg_station_balance │
                                           └──────────────────────┘
                                           ┌──────────────────────┐
                                           │  agg_weather_impact  │
                                           └──────────────────────┘
```

| Tabela Gold | Grão | Uso principal |
|---|---|---|
| `dim_time` | 1 linha por hora | Calendar lookup para joins |
| `dim_stations` | 1 linha por estação | Referência enriquecida |
| `fact_trips` | 1 linha por viagem | Tabela fato central (BI) |
| `agg_demand_hourly` | estação × hora × data | Modelo preditivo de demanda |
| `agg_station_balance` | estação × dia | Rebalanceamento de frota |
| `agg_weather_impact` | condição climática × hora | Correlação chuva/temperatura |


## 0. Setup & Dependências

In [1]:
!pip install pyspark delta-spark pyarrow pandas -q

In [2]:
import os
import sys
import platform
import logging
from pathlib import Path
from datetime import datetime, timedelta, date

import pandas as pd
import pyarrow as pa

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, TimestampType,
    IntegerType, LongType, BooleanType, DateType, FloatType
)
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('citibike_gold')
print(f"Python {sys.version}")
print(f"PyArrow {pa.__version__}")

Python 3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 10:40:35) [GCC 12.3.0]
PyArrow 13.0.0


In [3]:
# ============================================================
# CONFIGURACAO — Caminhos do Data Lake
# ============================================================

PROJECT_ROOT = Path(os.getcwd()).parent
DATA_DIR     = PROJECT_ROOT / 'dados'

SILVER_DIR = DATA_DIR / 'silver'
GOLD_DIR   = DATA_DIR / 'gold'

# Silver — entradas
SILVER_TRIPS    = SILVER_DIR / 'trips'
SILVER_WEATHER  = SILVER_DIR / 'weather'
SILVER_STATIONS = SILVER_DIR / 'stations'

# Gold — saídas Delta
GOLD_DIM_TIME       = GOLD_DIR / 'dim_time'
GOLD_DIM_STATIONS   = GOLD_DIR / 'dim_stations'
GOLD_FACT_TRIPS     = GOLD_DIR / 'fact_trips'
GOLD_AGG_DEMAND     = GOLD_DIR / 'agg_demand_hourly'
GOLD_AGG_BALANCE    = GOLD_DIR / 'agg_station_balance'
GOLD_AGG_WEATHER    = GOLD_DIR / 'agg_weather_impact'

for d in [GOLD_DIM_TIME, GOLD_DIM_STATIONS, GOLD_FACT_TRIPS,
          GOLD_AGG_DEMAND, GOLD_AGG_BALANCE, GOLD_AGG_WEATHER]:
    d.mkdir(parents=True, exist_ok=True)

# Verificar que Silver existe
for p in [SILVER_TRIPS, SILVER_WEATHER, SILVER_STATIONS]:
    if not any(p.rglob('*.parquet')):
        raise FileNotFoundError(
            f"Silver vazio: {p}\nExecute o Notebook 03 antes de continuar."
        )
    print(f"  Silver OK: {p.name}")
print("Paths configurados.")

  Silver OK: trips
  Silver OK: weather
  Silver OK: stations
Paths configurados.


In [4]:
# ============================================================
# Inicializar SparkSession com Delta Lake
# ============================================================

os.environ['PYSPARK_PYTHON']        = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

if platform.system() == 'Windows':
    _hadoop_home = os.environ.get('HADOOP_HOME', '')
    if _hadoop_home:
        _hadoop_bin = os.path.join(_hadoop_home, 'bin')
        if _hadoop_bin not in os.environ.get('PATH', ''):
            os.environ['PATH'] = _hadoop_bin + ';' + os.environ.get('PATH', '')

_active = SparkSession.getActiveSession()
if _active is not None:
    _active.stop()

spark = configure_spark_with_delta_pip(
    SparkSession.builder
    .appName('CitiBike_Gold')
    .config('spark.driver.memory', '8g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
    .config('spark.sql.warehouse.dir',
            str(DATA_DIR / 'spark-warehouse').replace('\\', '/'))
    .config('spark.local.dir',
            str(DATA_DIR / 'spark-temp').replace('\\', '/'))
    .config('spark.databricks.delta.retentionDurationCheck.enabled', 'false')
    .config('spark.sql.parquet.compression.codec', 'snappy')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .master('local[*]')
).getOrCreate()

if platform.system() == 'Windows':
    _hadoop_home = os.environ.get('HADOOP_HOME', '')
    if _hadoop_home:
        spark._jvm.System.setProperty(
            'hadoop.home.dir', _hadoop_home.replace('\\', '/'))

spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} + Delta Lake inicializado")
print(f"  Cores: {spark.sparkContext.defaultParallelism}")

Spark 3.5.0 + Delta Lake inicializado
  Cores: 24


---
## 1. Carregar Camada Silver

Leitura das três tabelas Delta da camada Silver como base para todas as transformações Gold.


In [5]:
# ============================================================
# 1.1 Carregar Silver Trips, Weather e Stations
# ============================================================

def silver_path(p):
    return str(p).replace('\\', '/')

df_trips    = spark.read.format('delta').load(silver_path(SILVER_TRIPS))
df_weather  = spark.read.format('delta').load(silver_path(SILVER_WEATHER))
df_stations = spark.read.format('delta').load(silver_path(SILVER_STATIONS))

print(f"Silver Trips:    {df_trips.count():>10,} registros")
print(f"Silver Weather:  {df_weather.count():>10,} registros")
print(f"Silver Stations: {df_stations.count():>10,} estações")

# Registrar como views SQL para uso ao longo do notebook
df_trips.createOrReplaceTempView('silver_trips')
df_weather.createOrReplaceTempView('silver_weather')
df_stations.createOrReplaceTempView('silver_stations')
print("\nViews SQL registradas.")

Silver Trips:    92,804,647 registros
Silver Weather:      27,720 registros
Silver Stations:      2,360 estações

Views SQL registradas.


---
## 2. `dim_time` — Dimensão de Calendário

Tabela de referência com **uma linha por hora** cobrindo todo o período dos dados.
Usada em joins para enriquecer agregações com atributos de tempo.

Colunas geradas: `datetime_hour`, `date`, `year`, `month`, `day`, `hour_of_day`,
`day_of_week`, `day_name`, `is_weekend`, `season`, `quarter`, `week_of_year`.


In [6]:
# ============================================================
# 2.1 Gerar dim_time (cobrindo o período dos dados)
# ============================================================

# Descobrir intervalo real dos dados Silver
date_range = df_trips.select(
    F.min('started_at').alias('min_dt'),
    F.max('started_at').alias('max_dt')
).collect()[0]

start_dt = date_range['min_dt'].replace(minute=0, second=0, microsecond=0)
end_dt   = date_range['max_dt'].replace(minute=0, second=0, microsecond=0)

print(f"Período dos dados: {start_dt}  →  {end_dt}")

# Gerar sequência horária via Pandas (mais rápido para séries de tempo)
import pandas as pd
hours = pd.date_range(start=start_dt, end=end_dt, freq='h')

pdf_time = pd.DataFrame({'datetime_hour': hours})
pdf_time['date']         = pdf_time['datetime_hour'].dt.date
pdf_time['year']         = pdf_time['datetime_hour'].dt.year
pdf_time['month']        = pdf_time['datetime_hour'].dt.month
pdf_time['day']          = pdf_time['datetime_hour'].dt.day
pdf_time['hour_of_day']  = pdf_time['datetime_hour'].dt.hour
pdf_time['day_of_week']  = pdf_time['datetime_hour'].dt.dayofweek  # 0=Seg, 6=Dom
pdf_time['day_name']     = pdf_time['datetime_hour'].dt.strftime('%A')
pdf_time['is_weekend']   = pdf_time['day_of_week'].isin([5, 6])
pdf_time['quarter']      = pdf_time['datetime_hour'].dt.quarter
pdf_time['week_of_year'] = pdf_time['datetime_hour'].dt.isocalendar().week.astype(int)

# Estação do ano (hemisfério norte)
month_to_season = {12:'Winter',1:'Winter',2:'Winter',
                   3:'Spring',4:'Spring',5:'Spring',
                   6:'Summer',7:'Summer',8:'Summer',
                   9:'Fall',10:'Fall',11:'Fall'}
pdf_time['season'] = pdf_time['month'].map(month_to_season)

df_dim_time = spark.createDataFrame(pdf_time)
print(f"dim_time: {df_dim_time.count():,} horas geradas")
df_dim_time.show(3, truncate=False)

Período dos dados: 2023-12-31 13:00:00  →  2026-02-28 23:00:00
dim_time: 18,971 horas geradas
+-------------------+----------+----+-----+---+-----------+-----------+--------+----------+-------+------------+------+
|datetime_hour      |date      |year|month|day|hour_of_day|day_of_week|day_name|is_weekend|quarter|week_of_year|season|
+-------------------+----------+----+-----+---+-----------+-----------+--------+----------+-------+------------+------+
|2023-12-31 13:00:00|2023-12-31|2023|12   |31 |13         |6          |Sunday  |true      |4      |52          |Winter|
|2023-12-31 14:00:00|2023-12-31|2023|12   |31 |14         |6          |Sunday  |true      |4      |52          |Winter|
|2023-12-31 15:00:00|2023-12-31|2023|12   |31 |15         |6          |Sunday  |true      |4      |52          |Winter|
+-------------------+----------+----+-----+---+-----------+-----------+--------+----------+-------+------------+------+
only showing top 3 rows



In [7]:
# ============================================================
# 2.2 Escrever dim_time como Delta Lake
# ============================================================

gold_dim_time_path = str(GOLD_DIM_TIME).replace('\\', '/')

(
    df_dim_time
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .save(gold_dim_time_path)
)

DeltaTable.forPath(spark, gold_dim_time_path) \
    .history(1).select('version','timestamp','operation').show(truncate=False)

df_dim_time.createOrReplaceTempView('dim_time')
print(f"dim_time gravado: {str(GOLD_DIM_TIME)}")

+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|2      |2026-04-07 22:29:32.838|WRITE    |
+-------+-----------------------+---------+

dim_time gravado: /home/jovyan/work/dados/gold/dim_time


---
## 3. `dim_stations` — Dimensão de Estações Enriquecida

Combina a Silver Stations (referência geográfica) com métricas de uso históricas
calculadas sobre os trips: volume total, popularidade, fluxo médio diário.


In [8]:
# ============================================================
# 3.1 Calcular métricas de uso por estação
# ============================================================

# Departures por estação
df_departures = (
    df_trips
    .groupBy(F.col('start_station_id').alias('station_id'))
    .agg(
        F.count('*').alias('total_departures'),
        F.countDistinct('date').alias('active_days'),
        F.round(F.avg('trip_duration_sec') / 60, 1).alias('avg_trip_min_depart')
    )
)

# Arrivals por estação
df_arrivals = (
    df_trips
    .groupBy(F.col('end_station_id').alias('station_id'))
    .agg(F.count('*').alias('total_arrivals'))
)

# Combinar com Silver Stations
df_dim_stations = (
    df_stations
    .join(df_departures, 'station_id', how='left')
    .join(df_arrivals,   'station_id', how='left')
    .withColumn('total_trips',
        F.coalesce('total_departures', F.lit(0)) +
        F.coalesce('total_arrivals',   F.lit(0))
    )
    .withColumn('avg_daily_departures',
        F.round(
            F.coalesce('total_departures', F.lit(0)) /
            F.greatest(F.coalesce('active_days', F.lit(1)), F.lit(1)),
            1
        )
    )
    # Classificar popularidade em 3 tiers
    .withColumn('popularity_tier',
        F.when(F.col('total_trips') >= F.expr(
            '(SELECT percentile_approx(total_trips, 0.75) FROM (SELECT total_trips FROM '
            '(SELECT s.station_id, COALESCE(d.total_departures,0)+COALESCE(a.total_arrivals,0) '
            'AS total_trips FROM silver_stations s '
            'LEFT JOIN (SELECT start_station_id as station_id, count(*) as total_departures '
            'FROM silver_trips GROUP BY 1) d ON s.station_id=d.station_id '
            'LEFT JOIN (SELECT end_station_id as station_id, count(*) as total_arrivals '
            'FROM silver_trips GROUP BY 1) a ON s.station_id=a.station_id) t) t2)'
        ), F.lit('High'))
        .when(F.col('total_trips') >= F.expr(
            '(SELECT percentile_approx(total_trips, 0.25) FROM (SELECT total_trips FROM '
            '(SELECT s.station_id, COALESCE(d.total_departures,0)+COALESCE(a.total_arrivals,0) '
            'AS total_trips FROM silver_stations s '
            'LEFT JOIN (SELECT start_station_id as station_id, count(*) as total_departures '
            'FROM silver_trips GROUP BY 1) d ON s.station_id=d.station_id '
            'LEFT JOIN (SELECT end_station_id as station_id, count(*) as total_arrivals '
            'FROM silver_trips GROUP BY 1) a ON s.station_id=a.station_id) t) t2)'
        ), F.lit('Medium'))
        .otherwise(F.lit('Low'))
    )
    .withColumn('gold_ingestion_ts', F.current_timestamp())
)

print(f"dim_stations: {df_dim_stations.count():,} estações")
print("\nTop 10 estações por volume:")
df_dim_stations.select(
    'station_id','station_name','nyc_zone',
    'total_departures','total_arrivals','total_trips','popularity_tier'
).orderBy(F.desc('total_trips')).show(10, truncate=False)

dim_stations: 2,360 estações

Top 10 estações por volume:
+------------------------------------+--------------------------------------------+---------+----------------+--------------+-----------+---------------+
|station_id                          |station_name                                |nyc_zone |total_departures|total_arrivals|total_trips|popularity_tier|
+------------------------------------+--------------------------------------------+---------+----------------+--------------+-----------+---------------+
|026fd07d-d810-45dc-9237-a3d84dce4fa9|Bergen St & Mother Gaston Blvd              |Brooklyn |NULL            |NULL          |0          |High           |
|038b7369-9c6e-4eb2-8569-f7461443302f|Fulton St & Williams Ave                    |Brooklyn |NULL            |NULL          |0          |High           |
|053312f2-e77e-4674-b28b-ad357fbcb4ee|8 Ave & W 49 St                             |Manhattan|NULL            |NULL          |0          |High           |
|05d03a63-b5f2-40d

In [9]:
# ============================================================
# 3.2 Escrever dim_stations como Delta Lake
# ============================================================

gold_dim_stations_path = str(GOLD_DIM_STATIONS).replace('\\', '/')

(
    df_dim_stations
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .save(gold_dim_stations_path)
)

DeltaTable.forPath(spark, gold_dim_stations_path) \
    .history(1).select('version','timestamp','operation').show(truncate=False)

df_dim_stations.createOrReplaceTempView('dim_stations')
count = spark.read.format('delta').load(gold_dim_stations_path).count()
print(f"dim_stations gravado: {count:,} estações  |  {str(GOLD_DIM_STATIONS)}")

+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|2      |2026-04-07 22:29:57.024|WRITE    |
+-------+-----------------------+---------+

dim_stations gravado: 2,360 estações  |  /home/jovyan/work/dados/gold/dim_stations


---
## 4. `fact_trips` — Tabela Fato Central

Versão denormalizada e otimizada da Silver Trips para consumo analítico.
Aplicamos `OPTIMIZE + ZORDER BY (started_at, start_station_id)` para acelerar
consultas por período e por estação (os dois filtros mais comuns em BI).

Schema: subconjunto selecionado de colunas Silver + chave surrogate de data.


In [10]:
# ============================================================
# 4.1 Construir fact_trips
# ============================================================

df_fact_trips = (
    df_trips.select(
        # Chaves
        'ride_id',
        F.date_format('started_at', 'yyyyMMddHH').cast(LongType()).alias('time_key'),

        # Dimensões
        'rideable_type',
        'member_casual',
        'start_station_id',
        'end_station_id',
        'start_zone',
        'end_zone',

        # Tempo
        'started_at',
        'ended_at',
        'year',
        'month',
        'hour_of_day',
        'day_of_week',
        'is_weekend',
        'is_holiday',
        'season',

        # Métricas de viagem
        'trip_duration_sec',
        F.round(F.col('trip_duration_sec') / 60.0, 2).alias('trip_duration_min'),
        'distance_km',
        'avg_speed_kmh',

        # Contexto climático
        'weather_temp_c',
        'weather_precip_mm',
        'weather_is_raining',
        'weather_category',

        # Metadados
        F.current_timestamp().alias('gold_ingestion_ts')
    )
)

print(f"fact_trips: {df_fact_trips.count():,} registros")
df_fact_trips.printSchema()

fact_trips: 92,804,647 registros
root
 |-- ride_id: string (nullable = true)
 |-- time_key: long (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- member_casual: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_zone: string (nullable = true)
 |-- end_zone: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- is_weekend: boolean (nullable = true)
 |-- is_holiday: boolean (nullable = true)
 |-- season: string (nullable = true)
 |-- trip_duration_sec: long (nullable = true)
 |-- trip_duration_min: double (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- avg_speed_kmh: double (nullable = true)
 |-- weather_temp_c: double (nullable = true)
 |-- weather_p

In [11]:
# ============================================================
# 4.2 Escrever fact_trips como Delta Lake (particionado por year, month)
# ============================================================

gold_fact_trips_path = str(GOLD_FACT_TRIPS).replace('\\', '/')

(
    df_fact_trips
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .partitionBy('year', 'month')
    .save(gold_fact_trips_path)
)

# OPTIMIZE + ZORDER — compacta small files e ordena dados fisicamente
# para máxima performance em filtros por data e por estação
print("Executando OPTIMIZE + ZORDER BY (started_at, start_station_id)...")
spark.sql(f"""
    OPTIMIZE delta.`{gold_fact_trips_path}`
    ZORDER BY (started_at, start_station_id)
""")

DeltaTable.forPath(spark, gold_fact_trips_path) \
    .history(2).select('version','timestamp','operation','operationMetrics').show(truncate=False)

count = spark.read.format('delta').load(gold_fact_trips_path).count()
print(f"\nfact_trips gravado: {count:,} registros")
df_fact_trips.createOrReplaceTempView('fact_trips')

Executando OPTIMIZE + ZORDER BY (started_at, start_station_id)...
+-------+-----------------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation|operationMetrics                                                                                                                                                                                                                                                     |
+-------+-----------------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|3      |2026-04

---
## 5. `agg_demand_hourly` — Demanda Horária por Estação

Agregação principal para o **Modelo Preditivo de Demanda**.

Grão: `(start_station_id, date, hour_of_day)` — uma linha por estação por hora.

Features geradas para ML:
- `departures` / `arrivals` — volume de partidas e chegadas
- `net_flow` — `arrivals - departures` (positivo = estação acumulando bikes)
- `avg_trip_min` — duração média das viagens
- `pct_member` — proporção de usuários mensalistas
- `pct_electric` — proporção de bikes elétricas
- `weather_*` — contexto climático da hora


In [12]:
# ============================================================
# 5.1 Calcular departures por estação × hora
# ============================================================

df_departures_hourly = (
    df_trips
    .groupBy(
        F.col('start_station_id').alias('station_id'),
        'date',
        'hour_of_day',
        'year',
        'month',
        'day_of_week',
        'is_weekend',
        'is_holiday',
        'season',
        'weather_temp_c',
        'weather_precip_mm',
        'weather_is_raining',
        'weather_category'
    )
    .agg(
        F.count('*').alias('departures'),
        F.round(F.avg('trip_duration_sec') / 60, 2).alias('avg_trip_min'),
        F.round(
            F.sum(F.when(F.col('member_casual') == 'member', 1).otherwise(0)) /
            F.count('*') * 100, 1
        ).alias('pct_member'),
        F.round(
            F.sum(F.when(F.col('rideable_type') == 'electric_bike', 1).otherwise(0)) /
            F.count('*') * 100, 1
        ).alias('pct_electric'),
        F.round(F.avg('distance_km'), 3).alias('avg_distance_km')
    )
)

# ============================================================
# 5.2 Calcular arrivals por estação × hora
# ============================================================

df_arrivals_hourly = (
    df_trips
    .groupBy(
        F.col('end_station_id').alias('station_id'),
        'date',
        'hour_of_day'
    )
    .agg(F.count('*').alias('arrivals'))
)

# ============================================================
# 5.3 Combinar departures + arrivals → net_flow
# ============================================================

df_agg_demand = (
    df_departures_hourly
    .join(df_arrivals_hourly,
          on=['station_id', 'date', 'hour_of_day'],
          how='left')
    .withColumn('arrivals', F.coalesce('arrivals', F.lit(0)))
    .withColumn('net_flow',
        F.col('arrivals') - F.col('departures')   # positivo = acumulando bikes
    )
    # Pressão de demanda: z-score simplificado dentro da mesma hora do dia
    .withColumn('demand_pressure',
        F.round(
            (F.col('departures') - F.avg('departures').over(
                Window.partitionBy('hour_of_day', 'day_of_week')
            )) / F.stddev('departures').over(
                Window.partitionBy('hour_of_day', 'day_of_week')
            ),
            3
        )
    )
    .withColumn('gold_ingestion_ts', F.current_timestamp())
)

print(f"agg_demand_hourly: {df_agg_demand.count():,} registros")
df_agg_demand.show(5, truncate=False)

agg_demand_hourly: 20,361,288 registros
+----------+----------+-----------+----+-----+-----------+----------+----------+------+--------------+-----------------+------------------+----------------+----------+------------+----------+------------+---------------+--------+--------+---------------+-------------------------+
|station_id|date      |hour_of_day|year|month|day_of_week|is_weekend|is_holiday|season|weather_temp_c|weather_precip_mm|weather_is_raining|weather_category|departures|avg_trip_min|pct_member|pct_electric|avg_distance_km|arrivals|net_flow|demand_pressure|gold_ingestion_ts        |
+----------+----------+-----------+----+-----+-----------+----------+----------+------+--------------+-----------------+------------------+----------------+----------+------------+----------+------------+---------------+--------+--------+---------------+-------------------------+
|2377.01   |2025-12-12|0          |2025|12   |6          |false     |false     |Winter|-2.4          |0.0            

In [13]:
# ============================================================
# 5.4 Escrever agg_demand_hourly como Delta Lake
# ============================================================

gold_agg_demand_path = str(GOLD_AGG_DEMAND).replace('\\', '/')

(
    df_agg_demand
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .partitionBy('year', 'month')
    .save(gold_agg_demand_path)
)

# ZORDER por estação + data — consultas típicas de ML filtram por esses campos
spark.sql(f"""
    OPTIMIZE delta.`{gold_agg_demand_path}`
    ZORDER BY (station_id, date, hour_of_day)
""")

DeltaTable.forPath(spark, gold_agg_demand_path) \
    .history(2).select('version','timestamp','operation').show(truncate=False)

df_agg_demand.createOrReplaceTempView('agg_demand_hourly')
print(f"agg_demand_hourly gravado: {str(GOLD_AGG_DEMAND)}")

+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|3      |2026-04-07 22:38:59.704|OPTIMIZE |
|2      |2026-04-07 22:38:30.041|WRITE    |
+-------+-----------------------+---------+

agg_demand_hourly gravado: /home/jovyan/work/dados/gold/agg_demand_hourly


---
## 6. `agg_station_balance` — Balanço Diário por Estação

Grão: `(station_id, date)` — uma linha por estação por dia.

Métrica principal: `net_flow_day = arrivals_day - departures_day`
- **Positivo** → estação está acumulando bikes (precisa de retirada)
- **Negativo** → estação está perdendo bikes (precisa de reposição)
- **Cumulativo** → `cumulative_balance` (janela temporal) sinaliza necessidade de rebalanceamento


In [14]:
# ============================================================
# 6.1 Calcular balanço diário
# ============================================================

df_daily_dep = (
    df_trips
    .groupBy(F.col('start_station_id').alias('station_id'), 'date', 'year', 'month')
    .agg(F.count('*').alias('departures_day'))
)

df_daily_arr = (
    df_trips
    .groupBy(F.col('end_station_id').alias('station_id'), 'date')
    .agg(F.count('*').alias('arrivals_day'))
)

# Window para balanço cumulativo por estação (ordenado por data)
w_cumulative = (
    Window
    .partitionBy('station_id')
    .orderBy('date')
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_agg_balance = (
    df_daily_dep
    .join(df_daily_arr, on=['station_id', 'date'], how='left')
    .withColumn('arrivals_day',  F.coalesce('arrivals_day',  F.lit(0)))
    .withColumn('net_flow_day',
        F.col('arrivals_day') - F.col('departures_day')
    )
    # Balanço acumulado — sinaliza deriva de longo prazo
    .withColumn('cumulative_balance',
        F.sum('net_flow_day').over(w_cumulative)
    )
    # Flag de alerta: se acumulado > +30 ou < -30 bikes → rebalancear
    .withColumn('rebalance_alert',
        (F.abs('cumulative_balance') > 30).cast(BooleanType())
    )
    # Enriquecer com metadata da estação
    .join(
        df_stations.select('station_id','station_name','nyc_zone','capacity'),
        on='station_id', how='left'
    )
    .withColumn('gold_ingestion_ts', F.current_timestamp())
    .orderBy('station_id', 'date')
)

n_alerts = df_agg_balance.filter('rebalance_alert').count()
total    = df_agg_balance.count()
print(f"agg_station_balance: {total:,} registros")
print(f"Alertas de rebalanceamento: {n_alerts:,} ({n_alerts/total*100:.1f}%)")
df_agg_balance.show(5, truncate=False)

agg_station_balance: 1,673,842 registros
Alertas de rebalanceamento: 1,245,567 (74.4%)
+----------+----------+----+-----+--------------+------------+------------+------------------+---------------+------------+--------+--------+--------------------------+
|station_id|date      |year|month|departures_day|arrivals_day|net_flow_day|cumulative_balance|rebalance_alert|station_name|nyc_zone|capacity|gold_ingestion_ts         |
+----------+----------+----+-----+--------------+------------+------------+------------------+---------------+------------+--------+--------+--------------------------+
|1234.56   |2025-06-18|2025|6    |1             |2           |1           |1                 |false          |NULL        |NULL    |NULL    |2026-04-07 22:39:16.873612|
|1234.56   |2025-06-19|2025|6    |1             |0           |-1          |0                 |false          |NULL        |NULL    |NULL    |2026-04-07 22:39:16.873612|
|1234.56   |2025-06-20|2025|6    |7             |6           |-1    

In [15]:
# ============================================================
# 6.2 Escrever agg_station_balance como Delta Lake
# ============================================================

gold_agg_balance_path = str(GOLD_AGG_BALANCE).replace('\\', '/')

(
    df_agg_balance
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .partitionBy('year', 'month')
    .save(gold_agg_balance_path)
)

spark.sql(f"""
    OPTIMIZE delta.`{gold_agg_balance_path}`
    ZORDER BY (station_id, date)
""")

DeltaTable.forPath(spark, gold_agg_balance_path) \
    .history(2).select('version','timestamp','operation').show(truncate=False)

df_agg_balance.createOrReplaceTempView('agg_station_balance')
print(f"agg_station_balance gravado: {str(GOLD_AGG_BALANCE)}")

+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|3      |2026-04-07 22:39:50.623|OPTIMIZE |
|2      |2026-04-07 22:39:41.323|WRITE    |
+-------+-----------------------+---------+

agg_station_balance gravado: /home/jovyan/work/dados/gold/agg_station_balance


---
## 7. `agg_weather_impact` — Impacto do Clima na Demanda

Grão: `(weather_category, hour_of_day, is_weekend)` — perfil de demanda por condição climática.

Responde: *"Chuva às 8h numa sexta reduz quantos % das viagens?"*
Usado para ajuste de previsão no modelo preditivo.


In [16]:
# ============================================================
# 7.1 Calcular impacto climático
# ============================================================

# Demanda média por hora em dias claros (baseline)
df_baseline = (
    df_trips
    .filter(F.col('weather_category') == 'Clear')
    .groupBy('hour_of_day', 'is_weekend')
    .agg(
        F.count('*').alias('_total_clear'),
        F.countDistinct('date').alias('_days_clear')
    )
    .withColumn('avg_demand_clear',
        F.round(F.col('_total_clear') / F.col('_days_clear'), 2)
    )
    .select('hour_of_day', 'is_weekend', 'avg_demand_clear')
)

# Demanda média por hora para cada categoria climática
df_by_weather = (
    df_trips
    .filter(F.col('weather_category').isNotNull())
    .groupBy('weather_category', 'hour_of_day', 'is_weekend')
    .agg(
        F.count('*').alias('total_trips'),
        F.countDistinct('date').alias('n_days'),
        F.round(F.avg('trip_duration_sec') / 60, 2).alias('avg_duration_min'),
        F.round(F.avg('distance_km'), 3).alias('avg_distance_km'),
        F.round(F.avg('weather_temp_c'), 1).alias('avg_temp_c'),
        F.round(F.avg('weather_precip_mm'), 2).alias('avg_precip_mm'),
        F.round(
            F.sum(F.when(F.col('member_casual') == 'member', 1).otherwise(0)) /
            F.count('*') * 100, 1
        ).alias('pct_member')
    )
    .withColumn('avg_demand',
        F.round(F.col('total_trips') / F.col('n_days'), 2)
    )
)

# Join com baseline para calcular impacto relativo
df_agg_weather = (
    df_by_weather
    .join(df_baseline, on=['hour_of_day', 'is_weekend'], how='left')
    .withColumn('demand_vs_clear_pct',
        F.round(
            (F.col('avg_demand') - F.col('avg_demand_clear')) /
            F.col('avg_demand_clear') * 100,
            1
        )
    )
    .drop('avg_demand_clear')
    .withColumn('gold_ingestion_ts', F.current_timestamp())
    .orderBy('hour_of_day', 'weather_category')
)

print(f"agg_weather_impact: {df_agg_weather.count():,} registros")
print("\nImpacto do clima vs dia claro (media todas as horas):")
df_agg_weather.groupBy('weather_category').agg(
    F.round(F.avg('demand_vs_clear_pct'), 1).alias('avg_impact_pct'),
    F.round(F.avg('avg_temp_c'), 1).alias('avg_temp_c'),
    F.round(F.avg('avg_precip_mm'), 2).alias('avg_precip_mm')
).orderBy('avg_impact_pct').show(truncate=False)

agg_weather_impact: 144 registros

Impacto do clima vs dia claro (media todas as horas):
+----------------+--------------+----------+-------------+
|weather_category|avg_impact_pct|avg_temp_c|avg_precip_mm|
+----------------+--------------+----------+-------------+
|snow            |NULL          |-0.5      |0.44         |
|clear           |NULL          |15.2      |0.0          |
|rain            |NULL          |17.3      |0.82         |
+----------------+--------------+----------+-------------+



In [17]:
# ============================================================
# 7.2 Escrever agg_weather_impact como Delta Lake
# ============================================================

gold_agg_weather_path = str(GOLD_AGG_WEATHER).replace('\\', '/')

(
    df_agg_weather
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .save(gold_agg_weather_path)
    # Sem partição — tabela pequena, lida inteira como lookup
)

DeltaTable.forPath(spark, gold_agg_weather_path) \
    .history(1).select('version','timestamp','operation').show(truncate=False)

df_agg_weather.createOrReplaceTempView('agg_weather_impact')
print(f"agg_weather_impact gravado: {str(GOLD_AGG_WEATHER)}")

+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|1      |2026-04-07 22:40:06.673|WRITE    |
+-------+-----------------------+---------+

agg_weather_impact gravado: /home/jovyan/work/dados/gold/agg_weather_impact


---
## 8. Consultas Analíticas — Showcase da Camada Gold

Demonstração das tabelas Gold via Spark SQL, simulando queries de BI e ML.


In [18]:
# ============================================================
# 8.1 Top 10 rotas (origem → destino) por volume
# ============================================================

print("=== TOP 10 ROTAS ===")
spark.sql("""
    SELECT
        s_start.station_name AS origem,
        s_end.station_name   AS destino,
        s_start.nyc_zone     AS zona_origem,
        COUNT(*)             AS total_viagens,
        ROUND(AVG(f.trip_duration_min), 1) AS duracao_media_min,
        ROUND(AVG(f.distance_km), 2)       AS distancia_media_km
    FROM fact_trips f
    JOIN dim_stations s_start ON f.start_station_id = s_start.station_id
    JOIN dim_stations s_end   ON f.end_station_id   = s_end.station_id
    WHERE f.start_station_id != f.end_station_id
    GROUP BY 1, 2, 3
    ORDER BY total_viagens DESC
    LIMIT 10
""").show(truncate=False)

=== TOP 10 ROTAS ===
+------+-------+-----------+-------------+-----------------+------------------+
|origem|destino|zona_origem|total_viagens|duracao_media_min|distancia_media_km|
+------+-------+-----------+-------------+-----------------+------------------+
+------+-------+-----------+-------------+-----------------+------------------+



In [19]:
# ============================================================
# 8.2 Perfil de demanda por hora do dia (membro vs casual)
# ============================================================

print("=== DEMANDA POR HORA — MEMBRO vs CASUAL ===")
spark.sql("""
    SELECT
        hour_of_day,
        ROUND(SUM(CASE WHEN member_casual='member' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1)
            AS pct_member,
        ROUND(SUM(CASE WHEN weather_is_raining THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1)
            AS pct_raining,
        COUNT(*) AS total_trips,
        ROUND(AVG(trip_duration_min), 1) AS avg_min,
        ROUND(AVG(distance_km), 2) AS avg_km
    FROM fact_trips
    GROUP BY hour_of_day
    ORDER BY hour_of_day
""").show(24, truncate=False)

=== DEMANDA POR HORA — MEMBRO vs CASUAL ===
+-----------+----------+-----------+-----------+-------+------+
|hour_of_day|pct_member|pct_raining|total_trips|avg_min|avg_km|
+-----------+----------+-----------+-----------+-------+------+
|0          |74.2      |5.3        |1296971    |12.9   |2.18  |
|1          |72.2      |6.2        |782920     |13.0   |2.16  |
|2          |71.4      |5.9        |501744     |13.1   |2.18  |
|3          |71.6      |6.4        |332448     |12.8   |2.15  |
|4          |75.5      |6.0        |309939     |11.7   |2.17  |
|5          |85.9      |4.6        |679432     |9.7    |2.03  |
|6          |89.2      |4.7        |1845100    |9.7    |2.02  |
|7          |89.9      |6.3        |3798699    |10.3   |2.04  |
|8          |90.1      |4.6        |5845676    |10.9   |2.04  |
|9          |87.0      |6.9        |4867203    |11.2   |1.94  |
|10         |82.6      |7.9        |4131693    |12.2   |1.93  |
|11         |80.3      |7.3        |4438206    |12.9   |1.92

In [20]:
# ============================================================
# 8.3 Estações com maior risco de esvaziamento
# ============================================================

print("=== ESTAÇÕES COM MAIOR RISCO DE ESVAZIAMENTO (últimos 30 dias) ===")
spark.sql("""
    WITH max_date AS (
        SELECT MAX(date) AS max_dt FROM agg_station_balance
    )
    SELECT
        b.station_id,
        b.station_name,
        b.nyc_zone,
        b.capacity,
        SUM(b.departures_day)     AS total_departures,
        SUM(b.arrivals_day)       AS total_arrivals,
        SUM(b.net_flow_day)       AS net_flow_periodo,
        MIN(b.cumulative_balance) AS min_balance,
        MAX(b.rebalance_alert)    AS teve_alerta
    FROM agg_station_balance b, max_date m
    WHERE b.date >= DATE_SUB(m.max_dt, 30)
    GROUP BY 1,2,3,4
    HAVING net_flow_periodo < -50
    ORDER BY net_flow_periodo ASC
    LIMIT 15
""").show(truncate=False)

=== ESTAÇÕES COM MAIOR RISCO DE ESVAZIAMENTO (últimos 30 dias) ===
+----------+------------+--------+--------+----------------+--------------+----------------+-----------+-----------+
|station_id|station_name|nyc_zone|capacity|total_departures|total_arrivals|net_flow_periodo|min_balance|teve_alerta|
+----------+------------+--------+--------+----------------+--------------+----------------+-----------+-----------+
|6364.10   |NULL        |NULL    |NULL    |2949            |1954          |-995            |-36845     |true       |
|7154.10   |NULL        |NULL    |NULL    |1544            |865           |-679            |-39847     |true       |
|5343.10   |NULL        |NULL    |NULL    |1905            |1280          |-625            |-40185     |true       |
|7756.10   |NULL        |NULL    |NULL    |1293            |699           |-594            |-17705     |true       |
|5997.10   |NULL        |NULL    |NULL    |1310            |854           |-456            |-34977     |true      

In [21]:
# ============================================================
# 8.4 Impacto da chuva nos horários de pico
# ============================================================

print("=== IMPACTO DA CHUVA NOS HORÁRIOS DE PICO (7h-10h e 17h-20h) ===")
spark.sql("""
    SELECT
        weather_category,
        is_weekend,
        ROUND(AVG(demand_vs_clear_pct), 1)  AS reducao_vs_dia_claro_pct,
        ROUND(AVG(avg_temp_c), 1)           AS temp_media_c,
        ROUND(AVG(avg_precip_mm), 2)        AS precip_media_mm,
        ROUND(AVG(avg_duration_min), 1)     AS duracao_media_min
    FROM agg_weather_impact
    WHERE hour_of_day BETWEEN 7 AND 10
       OR hour_of_day BETWEEN 17 AND 20
    GROUP BY 1, 2
    ORDER BY reducao_vs_dia_claro_pct ASC
""").show(truncate=False)

=== IMPACTO DA CHUVA NOS HORÁRIOS DE PICO (7h-10h e 17h-20h) ===
+----------------+----------+------------------------+------------+---------------+-----------------+
|weather_category|is_weekend|reducao_vs_dia_claro_pct|temp_media_c|precip_media_mm|duracao_media_min|
+----------------+----------+------------------------+------------+---------------+-----------------+
|rain            |false     |NULL                    |16.9        |0.88           |11.0             |
|rain            |true      |NULL                    |17.5        |0.73           |12.4             |
|snow            |false     |NULL                    |-1.5        |0.34           |9.7              |
|clear           |true      |NULL                    |15.6        |0.0            |13.2             |
|snow            |true      |NULL                    |-0.9        |0.53           |9.6              |
|clear           |false     |NULL                    |15.0        |0.0            |11.7             |
+----------------

In [22]:
# ============================================================
# 8.5 Delta Lake — Inventário completo da camada Gold
# ============================================================

print("=== INVENTÁRIO GOLD LAYER ===")
gold_tables = {
    'dim_time':            str(GOLD_DIM_TIME).replace('\\', '/'),
    'dim_stations':        str(GOLD_DIM_STATIONS).replace('\\', '/'),
    'fact_trips':          str(GOLD_FACT_TRIPS).replace('\\', '/'),
    'agg_demand_hourly':   str(GOLD_AGG_DEMAND).replace('\\', '/'),
    'agg_station_balance': str(GOLD_AGG_BALANCE).replace('\\', '/'),
    'agg_weather_impact':  str(GOLD_AGG_WEATHER).replace('\\', '/')
}

print(f"{'Tabela':<25} {'Registros':>12}  {'Versão':>7}  Path")
print('-' * 90)
for name, path in gold_tables.items():
    try:
        cnt = spark.read.format('delta').load(path).count()
        ver = DeltaTable.forPath(spark, path).history(1).collect()[0]['version']
        print(f"{name:<25} {cnt:>12,}  {ver:>7}  {path}")
    except Exception as e:
        print(f"{name:<25} {'ERRO':>12}  {'—':>7}  {e}")

=== INVENTÁRIO GOLD LAYER ===
Tabela                       Registros   Versão  Path
------------------------------------------------------------------------------------------
dim_time                        18,971        2  /home/jovyan/work/dados/gold/dim_time
dim_stations                     2,360        2  /home/jovyan/work/dados/gold/dim_stations
fact_trips                  92,804,647        3  /home/jovyan/work/dados/gold/fact_trips
agg_demand_hourly           20,361,288        3  /home/jovyan/work/dados/gold/agg_demand_hourly
agg_station_balance          1,673,842        3  /home/jovyan/work/dados/gold/agg_station_balance
agg_weather_impact                 144        1  /home/jovyan/work/dados/gold/agg_weather_impact


---
## 9. Resumo da Camada Gold

### Tabelas entregues

| Tabela | Grão | Partição | ZORDER | Uso |
|---|---|---|---|---|
| `dim_time` | hora | — | — | Calendar lookup |
| `dim_stations` | estação | — | — | Dimensão geográfica + popularidade |
| `fact_trips` | viagem | `year, month` | `started_at, start_station_id` | BI / exploração |
| `agg_demand_hourly` | estação × hora × data | `year, month` | `station_id, date, hour_of_day` | Modelo preditivo de demanda |
| `agg_station_balance` | estação × dia | `year, month` | `station_id, date` | Rebalanceamento de frota |
| `agg_weather_impact` | clima × hora | — | — | Ajuste de previsão por condição climática |

### Garantias técnicas

- **ACID** em todas as escritas via Delta Lake
- **OPTIMIZE + ZORDER** nas tabelas maiores — aceleração de 10–100× em filtros comuns
- **Time travel** disponível em todas as tabelas (`versionAsOf` / `timestampAsOf`)
- **Schema enforcement** — rejeição automática de dados fora do schema

### Pipeline Medallion completo

```
Fontes (S3, API)
     ↓ Notebook 01
  Bronze (Parquet raw, sem transformação)
     ↓ Notebook 03
  Silver (Delta — limpo, enriquecido, deduplicado)
     ↓ Notebook 04
  Gold (Delta — agregado, otimizado, pronto para ML e BI)
```
